# Lab 2  Messy Data & Leakage-Safe Features

**DSA 8401 Applied Machine Learning · Chapter 2**

This notebook works the four Lab 2 tasks on the synthetic mobile-money dump `mobile_money_statements.csv` using the patterns from Chapter 2

1. **Audit**  profile missingness / duplicates / validity violations; classify each
   incomplete column MCAR / MAR / MNAR 
2. **Clean & aggregate**  parse amounts and dates
     - resolve entities to a canonical `customer_id` 
     -  build customer-level RFM, ratio and cyclical features 
3. **Hunt the leak**  correlation screen (Listing 2.11) + a lineage argument to find the two planted leaks.
4. **Pipeline**  one `ColumnTransformer` `Pipeline`, group-aware CV, ROC–AUC with and without the leaks 



In [1]:
import re, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")



In [2]:
# The scoring timestamp: every feature is computed over data strictly BEFORE this
SCORING_TS = pd.Timestamp("2026-08-01 00:00:00")



In [3]:
raw = pd.read_csv(r"../Data/mobile_money_statements.csv")
print("raw shape:", raw.shape)
raw.head(3)

raw shape: (183600, 18)


,txn_id,msisdn,reg_id,account_name,region,segment,txn_time,txn_type,amount,gps_lat,gps_lon,agent_id,device_id,counterparty,balance_after,manual_review_score,settlement_status,is_fraud
0,TXN0102856,254700004022,NID101089,Joseph Mwangi,Kampala,retail,2026-06-30 14:24:19,send,"1,329/- Dr",NaN,NaN,AG1014,DV978723,CP6324,8123.0,0.061,settled,0
1,TXN0155490,254700013612,NID103658,Rehema Ndlovu,Mombasa,retail,"Jan 29, 2026 01:17 PM",send,289/- Dr,1.77470,34.70915,AG1084,DV616667,CP2831,6322.0,0.000,settled,0
2,TXN0019434,254700005862,nid101584,Mary Mutua,Arusha,retail,"Sep 16, 2025 12:55 AM",receive,"1,571/-",-3.74336,33.67018,AG1091,DV637734,CP167,15237.0,0.912,reversed,1


## 1. Audit(Missing  Nos)

Three cheap diagnostics :

- **quantify** the missing fraction per column, 
- **test** whether missingness is informative,
- **inspect co-missingness**. 

We also check duplicates and the validity rules the dump violates.

In [150]:
# 1a. Missingness fraction per column (Listing 2.1, step 1) ---


miss = raw.isna().mean().sort_values(ascending=False)

print("Missing fraction per column:")

print(miss[miss > 0].round(4).to_string())

Missing fraction per column:
gps_lon      0.3621
gps_lat      0.3621
agent_id     0.1369
device_id    0.0591


In [151]:
# 1b. Duplicates ---
# Exact row duplicates (client retries / log merges) -> drop_duplicates dispatches them.


print("exact duplicate rows :", int(raw.duplicated().sum()))

print("duplicate txn_id      :", int(raw['txn_id'].duplicated().sum()))


# txn_id repeats exactly on the duplicate rows they are true row copies, not new events.

exact duplicate rows : 3600
duplicate txn_id      : 3600


In [152]:
# 1c. Validity-rule violations ---

# amount is stored as a STRING with thousands separators, a "/-" suffix, and three
# different debit encodings; it will not cast to a number as-is.

print("amount dtype:", raw['amount'].dtype)

print("amount samples:", raw['amount'].dropna().sample(6, random_state=1).tolist())

# txn_time mixes three formats -> a single to_datetime(format=...) cannot parse all rows.
print("txn_time samples:", raw['txn_time'].sample(6, random_state=2).tolist())

amount dtype: object
amount samples: ['71/- Dr', '4,742/- Dr', '1,701/-', '(227/-)', '(165/-)', '(51/-)']
txn_time samples: ['29/06/2026 04:49', '2025-09-09 05:09:38', 'Mar 20, 2026 02:09 PM', 'May 04, 2025 08:43 PM', '2026-03-30 12:00:04', '03/08/2025 21:49']


In [153]:
raw['agent_id'].isna().astype(int)

0         0
1         0
2         0
3         0
4         1
         ..
183595    0
183596    1
183597    0
183598    0
183599    0
Name: agent_id, Length: 183600, dtype: int64

In [154]:
# -1d. Is missingness informative? (the MNAR screen)

# Compare the target rate across the missing indicator for each incomplete column.

for col in ["gps_lat", "agent_id", "device_id"]:

    # Fraud rate when the value is present
    present_rate = raw.loc[raw[col].notna(), "is_fraud"].mean()

    # Fraud rate when the value is missing
    missing_rate = raw.loc[raw[col].isna(), "is_fraud"].mean()

    print(f"\n{col}")
    print(f"  Present : {present_rate:.4f}")
    print(f"  Missing : {missing_rate:.4f}")


gps_lat
  Present : 0.0839
  Missing : 0.0843

agent_id
  Present : 0.0841
  Missing : 0.0839

device_id
  Present : 0.0840
  Missing : 0.0838


In [155]:
# 1e. Can OBSERVED columns explain the missingness? (MAR test) ---
# agent_id: missing fraction by region.


# Percentage of missing agent_id values in each region
print("Missing agent_id by region")
print(
    raw.groupby("region")["agent_id"]
       .apply(lambda x: x.isna().mean())
       .round(3)
)

# Percentage of missing gps_lat values in each region
print("\nMissing gps_lat by region")
print(
    raw.groupby("region")["gps_lat"]
       .apply(lambda x: x.isna().mean())
       .round(3)
)

# Percentage of missing device_id values in each region
print("\nMissing device_id by region")
print(
    raw.groupby("region")["device_id"]
       .apply(lambda x: x.isna().mean())
       .round(3)
)

Missing agent_id by region
region
Arusha           0.452
Dar es Salaam    0.000
Eldoret          0.000
Jinja            0.451
Kampala          0.000
Kigali           0.000
Mombasa          0.000
Mwanza           0.448
Nairobi          0.000
Nakuru           0.000
Name: agent_id, dtype: float64

Missing gps_lat by region
region
Arusha           0.360
Dar es Salaam    0.362
Eldoret          0.357
Jinja            0.365
Kampala          0.365
Kigali           0.367
Mombasa          0.354
Mwanza           0.364
Nairobi          0.360
Nakuru           0.366
Name: gps_lat, dtype: float64

Missing device_id by region
region
Arusha           0.060
Dar es Salaam    0.059
Eldoret          0.058
Jinja            0.056
Kampala          0.061
Kigali           0.061
Mombasa          0.060
Mwanza           0.058
Nairobi          0.059
Nakuru           0.058
Name: device_id, dtype: float64


### Step 1  findings and MCAR / MAR / MNAR classification

- MAR -  The missing value depends on another variable that you already have. , yes we can explain
- MNAR-  Missing because of the value itself ,No we cant
- MCAR  - Completely random, No we cant The missing values happened purely by chance.


**Duplicates.** Exactly **3,600** rows are exact copies (the same `txn_id` repeats). They must be dropped *before* any split, or the same event lands in train and test( cause leakage)

**Validity violations.** `amount` is a string (`"1,500/-"`, `"(227/-)"`, `"4,742/- Dr"`) three different debit encodings; `txn_time` mixes ISO, `dd/mm/yyyy`, and `Mon dd, yyyy` formats. Both need parsing before use.

**Missingness (evidence  mechanism).** Recall Definitions 
 - MCAR = missingness independent of everything
 - MAR = fully explained by *observed* columns; 
 - MNAR = depends on the *unobserved value itself*.


| Column | Missing | Evidence | Class |
|---|---|---|---|
| `agent_id` | ~14% | Missing fraction is ~45% in **Mwanza / Jinja / Arusha** and ~0% everywhere else — missingness is **fully explained by the observed `region`** column (a legacy logger in those regions). | **MAR** |
| `device_id` | ~6% | Missing fraction is flat across region/type and the missing-indicator does **not** move the fraud rate; there is no mechanism tying it to any value. Uniform, uninformative dropout. | **MCAR** |
| `gps_lat` / `gps_lon` | ~36% | Flat across observed columns too (so **not** MAR-explainable), but network/GPS coverage is worse in remote locations, i.e. the probability a coordinate is missing **depends on the coordinate itself**. | **MNAR** |


## 2. Clean and aggregate

- Deduplicate, 
- parse amounts and dates, 
- resolve entities to a canonical `customer_id`
- build customer-level RFM / ratio / cyclical features aligned to `SCORING_TS` (Eq. 2.1, Table 2.2, Listing 2.9).

In [156]:
#- 2a. Exact deduplication 
df = raw.drop_duplicates().reset_index(drop=True)
print("after dedup:", df.shape)

after dedup: (180000, 18)


In [157]:
# -2b. Parse the string amount into a signed numeric amount ---

# debit if it starts with '-', is wrapped in (), or is tagged ' Dr'; credit otherwise.

def parse_amount(s):
    
    s = str(s).strip()
    
    neg = s.startswith("(") or s.startswith("-") or "Dr" in s
    
    digits = re.sub(r"[^0-9]", "", s)          # strip commas, '/-', parens, 'Dr', sign
    
    if digits == "":
        return np.nan
    v = float(digits)
    return -v if neg else v



In [158]:
df["amount_signed"]    = df["amount"].map(parse_amount)  
df["amount_abs"]       = df["amount_signed"].abs()



print("amount parse failures:", int(df["amount_signed"].isna().sum()))
df[["amount", "amount_signed"]].head(4)

amount parse failures: 0


,amount,amount_signed
0,"1,329/- Dr",-1329.0
1,289/- Dr,-289.0
2,"1,571/-",1571.0
3,599/- Dr,-599.0


In [159]:
# Convert the transaction time column to text
s = df["txn_time"].astype(str)

# Try parsing the dates using different formats

# Format 1: 2025-01-15 14:30:00
ts = pd.to_datetime(s, format="%Y-%m-%d %H:%M:%S", errors="coerce")

# Format 2: 15/01/2025 14:30
mask = ts.isna()
ts.loc[mask] = pd.to_datetime(
                s[mask],
                format="%d/%m/%Y %H:%M",
                errors="coerce"
            )

# Format 3: Jan 15, 2025 02:30 PM
mask = ts.isna()
ts.loc[mask] = pd.to_datetime(
                    s[mask],
                    format="%b %d, %Y %I:%M %p",
                    errors="coerce"
                )

# Save the parsed dates
df["ts"] = ts

# Check how many dates could not be parsed
print("Number of invalid dates:", df["ts"].isna().sum())

# Check that all transactions happened before the scoring date
print("All transactions before scoring date:",
      (df["ts"] < SCORING_TS).all())

Number of invalid dates: 0
All transactions before scoring date: True


In [160]:
df["reg_id"].nunique()

11722

In [161]:
# - 2d. Entity resolution canonical customer_id 


# The msisdn (SIM) is NOT the identity: multi-SIM customers own several. The KYC
# registration id (reg_id) is the linking key; every SIM a person registers shares it.
# It is lightly messy (case / whitespace), so canonicalise before grouping.


# Create a clean customer ID
df["customer_id"] = (
                df["reg_id"]
                .astype(str)      # Convert to text
                .str.strip()      # Remove spaces
                .str.upper()      # Convert to uppercase
            )

# Count unique values
print("Unique SIM numbers (msisdn):", df["msisdn"].nunique())
print("Unique raw registration IDs:", df["reg_id"].nunique())
print("Unique cleaned customer IDs:", df["customer_id"].nunique())

#   keying on msisdn (or raw reg_id) would badly over-count customers and scatter a
#   person's SIMs across CV folds (leakage cause 5). We group on customer_id instead.

Unique SIM numbers (msisdn): 4825
Unique raw registration IDs: 11722
Unique cleaned customer IDs: 3991


In [162]:
# The scoring timestamp: every feature is computed over data strictly BEFORE this
SCORING_TS = pd.Timestamp("2026-08-01 00:00:00")

In [164]:
# Keep only transactions before the scoring date
df = df[df["ts"] < SCORING_TS].copy()

# ----------------------------
# Create time-based features
# ----------------------------

# Extract the hour of the day (0–23)
df["hour"] = df["ts"].dt.hour

# Extract the day of the week (Monday=0, Sunday=6)
df["dow"] = df["ts"].dt.dayofweek

# Extract the day of the month (1–31)
day = df["ts"].dt.day

# ----------------------------
# Create cyclical features
# ----------------------------

# Hour of the day
df["hr_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hr_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

# Day of the week
df["dow_sin"] = np.sin(2 * np.pi * df["dow"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["dow"] / 7)

# ----------------------------
# Other useful features
# ----------------------------

# Distance from the nearest payday (1st or 28th of the month)
df["payday_dist"] = np.minimum((day - 1).abs(), (day - 28).abs())

# Reduce the effect of very large transaction amounts
df["log_amt"] = np.log1p(df["amount_abs"])

# Flag whether the GPS location is missing
df["gps_missing"] = df["gps_lat"].isna().astype(int)

In [165]:
# --- 2f. Customer-level RFM / ratio aggregates as-of SCORING_TS (Table 2.2) ---
# All aggregates use only rows with ts < SCORING_TS (enforced above), so no feature


is_debit  = df["amount_signed"] < 0
by        = df.groupby("customer_id")


agg     = pd.DataFrame({
                        "recency_days":   (SCORING_TS - by["ts"].max()).dt.total_seconds() / 86400,    # R
                        "frequency":      by.size(),                                                   # F
                        "monetary_out":   df.assign(o=np.where(is_debit,  df["amount_abs"], 0)).groupby("customer_id")["o"].sum(),  # M
                        "monetary_in":    df.assign(i=np.where(~is_debit, df["amount_abs"], 0)).groupby("customer_id")["i"].sum(),
                        "n_counterparty": by["counterparty"].nunique(),                                                          # stability: distinct counterparties
                        "cashout_ratio":  df.assign(c=(df["txn_type"]=="cashout").astype(int)).groupby("customer_id")["c"].mean(),
                    })


agg["out_in_ratio"] = agg["monetary_out"] / (agg["monetary_in"] + 1.0)  # scale-free ratio
df = df.merge(agg, on="customer_id", how="left")

print("modelling table:", df.shape, "| customers:", df["customer_id"].nunique())

df[["customer_id","recency_days","frequency","monetary_out","monetary_in",
    "n_counterparty","cashout_ratio","out_in_ratio"]].head(3)

modelling table: (180000, 38) | customers: 3991


,customer_id,recency_days,frequency,monetary_out,monetary_in,n_counterparty,cashout_ratio,out_in_ratio
0,NID101089,12.508333,125,158085.0,73259.0,123,0.136000,2.157862
1,NID103658,12.170995,59,60882.0,44395.0,58,0.118644,1.371340
2,NID101584,1.488889,31,33101.0,26404.0,31,0.129032,1.253588


## 3. Hunt the leak

A two-minute correlation screen  rank features by |ρ(f, y)| and interrogate
the top of the list. 

In [166]:
# --- 3a. Correlation screen over numeric candidates 
num_candidates = ["manual_review_score", "amount_abs", "log_amt",
                  "hr_sin", "hr_cos", "dow_sin", "dow_cos", "payday_dist", "gps_missing",
                  "recency_days", "frequency", "monetary_out", "monetary_in",
                  "n_counterparty", "cashout_ratio", "out_in_ratio"]


y = df["is_fraud"].values


corr = pd.Series({c: abs(np.corrcoef(df[c].fillna(df[c].median()), y)[0, 1]) for c in num_candidates}).sort_values(ascending=False)


print("|corr| with is_fraud:")
print(corr.round(3).to_string())

# manual_review_score ~0.98 <-- statistical smoke.

|corr| with is_fraud:
manual_review_score    0.981
hr_cos                 0.071
hr_sin                 0.032
log_amt                0.022
amount_abs             0.016
cashout_ratio          0.010
monetary_out           0.004
frequency              0.003
n_counterparty         0.003
monetary_in            0.003
payday_dist            0.002
out_in_ratio           0.002
dow_sin                0.002
recency_days           0.002
gps_missing            0.000
dow_cos                0.000


In [167]:
# --- 3b. The categorical suspect: settlement_status class separation ---

print("fraud rate by settlement_status:")
print(df.groupby("settlement_status")["is_fraud"].mean().round(3).to_string())

print("\ncounts:")
print(df["settlement_status"].value_counts().to_string())
# 'reversed' -> 100% fraud, 'held' -> ~65%, 'settled'/'pending' -> ~0%: near-perfect split.

fraud rate by settlement_status:
settlement_status
held        0.651
pending     0.000
reversed    1.000
settled     0.004

counts:
settlement_status
settled     154760
reversed      9903
pending       8241
held          7096


## 4. Pipeline

One `ColumnTransformer` + `Pipeline`  so every imputer / transform / encoder
is re-fit **inside** each CV fold  preprocessing leakage becomes structurally impossible.
Evaluated with `GroupKFold` grouped by the resolved `customer_id`. We report ROC–AUC **without** and **with** the leaky columns.

In [168]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, PowerTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, GroupKFold

NUMERICAL_COLS = ["amount_abs", "log_amt", "hr_sin", "hr_cos", "dow_sin", "dow_cos",
              "payday_dist", "gps_missing", "recency_days", "frequency",
              "monetary_out", "monetary_in", "n_counterparty", "cashout_ratio", "out_in_ratio"]

CATEGORY_COLS = ["txn_type", "region", "segment"]

LEAK_NUM   = ["manual_review_score"]     # <- to be dropped
LEAK_CAT   = ["settlement_status"]       # <- to be dropped

def build_model(num_cols, cat_cols):
    num = Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("power",  PowerTransformer(method="yeo-johnson")),  # transform to Gaussian-like for LR
        ("scale",  RobustScaler()),# scaling
    ])
    cat = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="UNK")),
        ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50)),
    ])
    prep = ColumnTransformer([("num", num, num_cols), ("cat", cat, cat_cols)])
    
    results= Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=2000))]) # Pipeline with preprocessing and classifier
    
    return results

y       = df["is_fraud"].values
groups  = df["customer_id"].values           # split by resolved customer, never by SIM
cv      = GroupKFold(n_splits=5)

In [169]:
# --- Honest model: leaky columns EXCLUDED ---


clean     = build_model(NUMERICAL_COLS, CATEGORY_COLS)
auc_clean = cross_val_score(clean, df, y, cv=cv, groups=groups, scoring="roc_auc")


print(f"ROC-AUC WITHOUT leaks: {auc_clean.mean():.4f}  (+/- {auc_clean.std():.4f})")

ROC-AUC WITHOUT leaks: 0.6252  (+/- 0.0030)


In [170]:
# --- Contaminated model: leaky columns INCLUDED ---

leaky     = build_model(NUMERICAL_COLS + LEAK_NUM, CATEGORY_COLS + LEAK_CAT)
auc_leaky = cross_val_score(leaky, df, y, cv=cv, groups=groups, scoring="roc_auc")


print(f"ROC-AUC WITH leaks   : {auc_leaky.mean():.4f}  (+/- {auc_leaky.std():.4f})")
print(f"Offline AUC inflation from leakage: {auc_leaky.mean() - auc_clean.mean():+.4f}")

ROC-AUC WITH leaks   : 1.0000  (+/- 0.0000)
Offline AUC inflation from leakage: +0.3748


### Step 4 comment on the gap

Including the two post-outcome fields lifts cross-validated ROC–AUC to **~1.00**, versus
**~0.63** for the honest feature set  an inflation of roughly **+0.37 AUC**. Every
offline safeguard (CV, held-out AUC, model comparison) rewards the leak, because on the
contaminated data the leaked columns genuinely predict the label. In production those
fields are written *after* the transaction is scored, so they would be absent (or null),
and the ~1.00 model would collapse toward the honest ~0.63 — or worse, since it learned to
lean on columns it can no longer see.

Two structural safeguards made the honest number trustworthy: (1) all preprocessing lives
**inside** the `Pipeline`, re-fit per fold, so no imputer/scaler statistic crosses the
train/validation boundary; and (2) `GroupKFold` on the resolved `customer_id` keeps every
SIM of a person on one side of each split (leakage cause 5). A leaky number is worth less
than an honest one.